In [0]:
# ============================================================
# UK RETAIL COMMERCIAL INTELLIGENCE PLATFORM
# MODULE 7 — ERP PRODUCT MASTER & BRONZE INGESTION
# Author: Byron Kamau
# ============================================================
# FILE STRUCTURES VERIFIED:
#   ons_retail_sales_index.xlsx  → sheet 'CPSA', header row 7, data from row 8
#   ons_retail_sales_timeseries.xlsx → sheet 'data', 622 series, rows=time periods
#   online_retail_II.xlsx → sheets 'Year 2009-2010' & 'Year 2010-2011'
#   Google Trends CSVs → skip 1 title row, then 2 columns (week/region, score)
# ============================================================

# Module 7 — Bronze Ingestion & ERP Foundation
**Layer:** Bronze → Silver (foundation tables)

| Table | Layer | Description |
|---|---|---|
| bronze_pos_raw | Bronze | All POS transactions (both Excel sheets unioned) |
| bronze_trends_baby_timeseries | Bronze | Weekly baby products search volume UK |
| bronze_trends_baby_regions | Bronze | Baby products interest by UK subregion |
| bronze_trends_nursery_timeseries | Bronze | Weekly nursery furniture search volume UK |
| bronze_trends_nursery_regions | Bronze | Nursery furniture interest by UK subregion |
| bronze_ons_retail_index | Bronze | ONS retail sales value index (CPSA sheet) |
| bronze_ons_retail_timeseries | Bronze | ONS monthly retail timeseries (key series only) |
| silver_erp_product_master | Silver | 40 SKUs: cost, RRP, category, supplier |
| silver_retailer_funding | Silver | 8 retailers: rebates, settlement, scan allowances |
| silver_promo_calendar | Silver | 12 promotional events with trade spend |
| silver_uk_region_lookup | Silver | City to UK region mapping table |

## Cell 1 — Install Dependencies

In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Cell 2 — Setup: Database & File Paths
⚠️ Update RAW_FILES_PATH to match where you uploaded your files in Databricks

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import numpy as np

# ── UPDATE THIS PATH ────────────────────────────────────────
# If using DBFS File Upload:  "/FileStore/uk_retail_project/"
# If using Unity Catalog:     "/Volumes/main/uk_retail/raw/"
RAW_FILES_PATH = "/Volumes/workspace/ukretailcommercialproject/uk_retail_project/"

DATABASE = "uk_retail_commercial"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE}")
spark.sql(f"USE {DATABASE}")

print(f"✅ Database ready  : {DATABASE}")
print(f"📁 Raw files path  : {RAW_FILES_PATH}")

✅ Database ready  : uk_retail_commercial
📁 Raw files path  : /Volumes/workspace/ukretailcommercialproject/uk_retail_project/


## Cell 3 — Ingest POS Transactions (Online Retail II)
Reads both sheets from `online_retail_II.xlsx` and unions into one Bronze table

In [0]:
file_pos = f"{RAW_FILES_PATH}online_retail_II.xlsx"

print("📖 Reading Year 2009-2010...")
df_2009_pd = pd.read_excel(file_pos, sheet_name="Year 2009-2010", dtype=str)
df_2009_pd["source_sheet"] = "2009-2010"
print(f"   Rows: {len(df_2009_pd):,}  |  Cols: {list(df_2009_pd.columns)}")

print("📖 Reading Year 2010-2011...")
df_2010_pd = pd.read_excel(file_pos, sheet_name="Year 2010-2011", dtype=str)
df_2010_pd["source_sheet"] = "2010-2011"
print(f"   Rows: {len(df_2010_pd):,}  |  Cols: {list(df_2010_pd.columns)}")

# Union both sheets
df_pos_pd = pd.concat([df_2009_pd, df_2010_pd], ignore_index=True)
print(f"\n✅ Combined rows: {len(df_pos_pd):,}")

# Convert to Spark and add ingestion timestamp
df_pos_raw = spark.createDataFrame(df_pos_pd)
df_pos_raw = df_pos_raw.withColumn("ingested_at", F.current_timestamp())

display(df_pos_raw.limit(5))

📖 Reading Year 2009-2010...
   Rows: 525,461  |  Cols: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'source_sheet']
📖 Reading Year 2010-2011...
   Rows: 541,910  |  Cols: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'source_sheet']

✅ Combined rows: 1,067,371


Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,ingested_at
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,2009-2010,2026-03-18T10:21:35.556Z
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-2010,2026-03-18T10:21:35.556Z
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-2010,2026-03-18T10:21:35.556Z
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.1,13085,United Kingdom,2009-2010,2026-03-18T10:21:35.556Z
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,2009-2010,2026-03-18T10:21:35.556Z


## Cell 4 — Write POS Bronze Delta Table

In [0]:
# Rename column with space before write
if 'Customer ID' in df_pos_raw.columns:
    df_pos_raw = df_pos_raw.withColumnRenamed('Customer ID', 'CustomerID')

(df_pos_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.bronze_pos_raw"))

n = spark.sql(f"SELECT COUNT(*) AS n FROM {DATABASE}.bronze_pos_raw").collect()[0]["n"]
print(f"✅ bronze_pos_raw written: {n:,} rows")

✅ bronze_pos_raw written: 1,067,371 rows


## Cell 5 — Ingest ONS Retail Sales Index
Sheet: `CPSA` — Value of retail sales at current prices, seasonally adjusted
Header is on row 7 (0-indexed = 6). Rows 8-9 are metadata codes, data starts row 10.

In [0]:
# CELL 5 FIX — Clean column names before writing to Delta
import re

file_ons_idx = f"{RAW_FILES_PATH}ons_retail_sales_index.xlsx"

df_ons_idx_pd = pd.read_excel(
    file_ons_idx,
    sheet_name="CPSA",
    header=6
)

# Drop the two metadata rows below the header
df_ons_idx_pd = df_ons_idx_pd.iloc[2:].reset_index(drop=True)

# Rename first column
df_ons_idx_pd = df_ons_idx_pd.rename(columns={"Time Period": "time_period"})

# Drop empty rows
df_ons_idx_pd = df_ons_idx_pd.dropna(subset=["time_period"])

# ── CLEAN COLUMN NAMES ──────────────────────────────────────
# Delta Lake doesn't allow: spaces , ; { } ( ) \n \t = [ ]
def clean_col(col):
    col = str(col)
    col = col.lower()
    col = re.sub(r'[\[\]]', '', col)         # remove square brackets
    col = re.sub(r'[,;{}()\n\t=]', '', col)  # remove other bad chars
    col = re.sub(r'\s+', '_', col)           # spaces → underscores
    col = re.sub(r'_+', '_', col)            # collapse multiple underscores
    col = col.strip('_')                     # strip leading/trailing underscores
    return col

df_ons_idx_pd.columns = [clean_col(c) for c in df_ons_idx_pd.columns]

print(f"✅ ONS Index rows: {len(df_ons_idx_pd):,}")
print("Cleaned columns:", list(df_ons_idx_pd.columns))

df_ons_idx = spark.createDataFrame(df_ons_idx_pd.astype(str))
df_ons_idx = df_ons_idx.withColumn("ingested_at", F.current_timestamp())

(df_ons_idx.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.bronze_ons_retail_index"))

print("✅ bronze_ons_retail_index written")
display(df_ons_idx.limit(5))

✅ ONS Index rows: 916
Cleaned columns: ['time_period', 'all_retailing_including_automotive_fuel_note1', 'all_retailing_excluding_automotive_fuel_note_1', 'predominantly_food_stores', 'total_of_predominantly_non-food_stores_note_2', 'non-specialised_stores_note_2', 'textile_clothing_and_footwear_stores_note_2', 'household_goods_stores_note_2', 'other_stores_note_2', 'non-store_retailing', 'predominantly_automotive_fuel_note_1']
✅ bronze_ons_retail_index written


time_period,all_retailing_including_automotive_fuel_note1,all_retailing_excluding_automotive_fuel_note_1,predominantly_food_stores,total_of_predominantly_non-food_stores_note_2,non-specialised_stores_note_2,textile_clothing_and_footwear_stores_note_2,household_goods_stores_note_2,other_stores_note_2,non-store_retailing,predominantly_automotive_fuel_note_1,ingested_at
1988 Jan,nan,27,23.3,34.9,30.1,32,48.6,32.3,15.3,nan,2026-03-18T10:32:39.510Z
1988 Feb,nan,26.6,23.2,34.2,30,31.5,47.5,31.1,14.8,nan,2026-03-18T10:32:39.510Z
1988 Mar,nan,26.6,23.1,34.2,30.1,31.4,47.8,31.2,14.9,nan,2026-03-18T10:32:39.510Z
1988 Apr,nan,27.3,23.6,35.1,30.4,31.3,48.8,33.1,15.9,nan,2026-03-18T10:32:39.510Z
1988 May,nan,27.3,23.5,35.3,30.4,32.1,49.4,32.7,15.6,nan,2026-03-18T10:32:39.510Z


## Cell 6 — Ingest ONS Retail Sales Timeseries
Sheet: `data` — 622 series columns, rows 8 onward = time periods
We extract only the 8 most relevant series for retail market context

In [0]:
import re

def clean_col(col):
    col = str(col).lower()
    col = re.sub(r'[\[\]]', '', col)
    col = re.sub(r'[,;{}()\n\t=]', '', col)
    col = re.sub(r'\s+', '_', col)
    col = re.sub(r'_+', '_', col)
    return col.strip('_')

file_ons_ts = f"{RAW_FILES_PATH}ons_retail_sales_timeseries.xlsx"

df_ts_full = pd.read_excel(file_ons_ts, sheet_name="data", header=None)

titles = df_ts_full.iloc[0].tolist()
cdids = df_ts_full.iloc[1].tolist()

target_cdids = ["J5C4", "J468", "EAQW", "EAQY", "EARA", "EARB", "EAQZ", "J5BI"]

selected_cols = [0]
for i, cdid in enumerate(cdids):
    if cdid in target_cdids:
        selected_cols.append(i)

df_ts_sel = df_ts_full.iloc[7:, selected_cols].copy()

col_names = ["time_period"] + [str(cdids[i]) for i in selected_cols[1:]]
df_ts_sel.columns = [clean_col(c) for c in col_names]

mask = df_ts_sel["time_period"].astype(str).str.match(r"^\d{4} [A-Za-z]{3}$")
df_ts_sel = df_ts_sel[mask].reset_index(drop=True)

print("Monthly rows:", len(df_ts_sel))
print("Columns:", list(df_ts_sel.columns))

df_ons_ts = spark.createDataFrame(df_ts_sel.astype(str))
df_ons_ts = df_ons_ts.withColumn("ingested_at", F.current_timestamp())

df_ons_ts.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DATABASE}.bronze_ons_retail_timeseries")

print("✅ bronze_ons_retail_timeseries written")
display(df_ons_ts.limit(5))

/local_disk0/.ephemeral_nfs/envs/pythonEnv-bd97007c-9a77-40e6-bac6-fefa6d9c8be1/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Monthly rows: 457
Columns: ['time_period', 'eaqw', 'eaqy', 'eaqz', 'eara', 'earb', 'j468', 'j5bi', 'j5c4']
✅ bronze_ons_retail_timeseries written


time_period,eaqw,eaqy,eaqz,eara,earb,j468,j5bi,j5c4,ingested_at
1988 JAN,23.3,34.9,32.3,32,48.6,27,15.3,nan,2026-03-18T10:38:29.975Z
1988 FEB,23.2,34.2,31.1,31.5,47.5,26.6,14.8,nan,2026-03-18T10:38:29.975Z
1988 MAR,23.1,34.2,31.2,31.4,47.8,26.6,14.9,nan,2026-03-18T10:38:29.975Z
1988 APR,23.6,35.1,33.1,31.3,48.8,27.3,15.9,nan,2026-03-18T10:38:29.975Z
1988 MAY,23.5,35.3,32.7,32.1,49.4,27.3,15.6,nan,2026-03-18T10:38:29.975Z


## Cell 7 — Ingest Google Trends: Baby Products (Time Series)

In [0]:
file_gt_baby = f"{RAW_FILES_PATH}google_trends_baby_products_UK.csv"

# Google Trends exports: row 1 = search label, row 2 = headers, row 3+ = data
df_baby_pd = pd.read_csv(file_gt_baby, skiprows=1)
print("Raw columns:", list(df_baby_pd.columns))
print(df_baby_pd.head(3).to_string())

df_baby_pd.columns        = ["week", "interest_score"]
df_baby_pd["search_term"] = "baby products"
df_baby_pd["geo"]         = "United Kingdom"

df_trends_baby = spark.createDataFrame(df_baby_pd.astype(str))
df_trends_baby = df_trends_baby.withColumn("ingested_at", F.current_timestamp())

(df_trends_baby.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.bronze_trends_baby_timeseries"))

print(f"✅ bronze_trends_baby_timeseries: {df_trends_baby.count()} rows")

Raw columns: ['Week', 'baby products: (United Kingdom)']
         Week  baby products: (United Kingdom)
0  2021-03-14                               33
1  2021-03-21                               35
2  2021-03-28                               36
✅ bronze_trends_baby_timeseries: 262 rows


## Cell 8 — Ingest Google Trends: Baby Products (Regions)

In [0]:
file_gt_baby_r = f"{RAW_FILES_PATH}google_trends_baby_products_UK_regions.csv"

df_baby_r_pd = pd.read_csv(file_gt_baby_r, skiprows=1)
df_baby_r_pd.columns        = ["region", "interest_score"]
df_baby_r_pd["search_term"] = "baby products"

df_trends_baby_r = spark.createDataFrame(df_baby_r_pd.astype(str))
df_trends_baby_r = df_trends_baby_r.withColumn("ingested_at", F.current_timestamp())

(df_trends_baby_r.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.bronze_trends_baby_regions"))

print("✅ bronze_trends_baby_regions written")
display(df_trends_baby_r)

✅ bronze_trends_baby_regions written


region,interest_score,search_term,ingested_at
England,100,baby products,2026-03-18T10:33:45.731Z
Northern Ireland,86,baby products,2026-03-18T10:33:45.731Z
Wales,82,baby products,2026-03-18T10:33:45.731Z
Scotland,70,baby products,2026-03-18T10:33:45.731Z


## Cell 9 — Ingest Google Trends: Nursery Furniture (Time Series)

In [0]:
file_gt_nur = f"{RAW_FILES_PATH}google_trends_nursery_furniture_UK.csv"

df_nur_pd = pd.read_csv(file_gt_nur, skiprows=1)
df_nur_pd.columns        = ["week", "interest_score"]
df_nur_pd["search_term"] = "nursery furniture"
df_nur_pd["geo"]         = "United Kingdom"

df_trends_nur = spark.createDataFrame(df_nur_pd.astype(str))
df_trends_nur = df_trends_nur.withColumn("ingested_at", F.current_timestamp())

(df_trends_nur.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.bronze_trends_nursery_timeseries"))

print(f"✅ bronze_trends_nursery_timeseries: {df_trends_nur.count()} rows")

✅ bronze_trends_nursery_timeseries: 262 rows


## Cell 10 — Ingest Google Trends: Nursery Furniture (Regions)

In [0]:
file_gt_nur_r = f"{RAW_FILES_PATH}google_trends_nursery_furniture_UK_regions.csv"

df_nur_r_pd = pd.read_csv(file_gt_nur_r, skiprows=1)
df_nur_r_pd.columns        = ["region", "interest_score"]
df_nur_r_pd["search_term"] = "nursery furniture"

df_trends_nur_r = spark.createDataFrame(df_nur_r_pd.astype(str))
df_trends_nur_r = df_trends_nur_r.withColumn("ingested_at", F.current_timestamp())

(df_trends_nur_r.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.bronze_trends_nursery_regions"))

print("✅ bronze_trends_nursery_regions written")
display(df_trends_nur_r)

✅ bronze_trends_nursery_regions written


region,interest_score,search_term,ingested_at
Northern Ireland,100,nursery furniture,2026-03-18T10:34:10.788Z
Wales,89,nursery furniture,2026-03-18T10:34:10.788Z
Scotland,82,nursery furniture,2026-03-18T10:34:10.788Z
England,79,nursery furniture,2026-03-18T10:34:10.788Z


## Cell 11 — Generate Synthetic ERP Product Master
40 SKUs across 8 nursery/baby categories

In [0]:
np.random.seed(42)

categories = {
    "Feeding":    ["Bottle Set", "Steriliser", "Weaning Spoons", "Sippy Cup", "Breast Pump"],
    "Travel":     ["Pushchair", "Car Seat", "Baby Carrier", "Travel Cot", "Buggy Board"],
    "Nursery":    ["Cot Bed", "Changing Mat", "Baby Monitor", "Night Light", "Moses Basket"],
    "Sleep":      ["Swaddle Blanket", "Sleep Bag", "White Noise Machine", "Cot Mobile", "Blackout Blind"],
    "Bathing":    ["Baby Bath", "Hooded Towel", "Bath Thermometer", "Bath Seat", "Baby Wash Set"],
    "Clothing":   ["Sleepsuit Set", "Scratch Mitts", "Baby Hat Set", "Bodysuit Pack", "Snowsuit"],
    "Toys":       ["Soft Rattle", "Activity Gym", "Stacking Rings", "Teething Toy", "Musical Box"],
    "Healthcare": ["Digital Thermometer", "Nasal Aspirator", "Nail File Set", "First Aid Kit", "Baby Scales"],
}

records = []
sku_id = 1000
for category, products in categories.items():
    for product in products:
        cost  = round(np.random.uniform(3.50, 45.00), 2)
        rrp   = round(cost * np.random.uniform(2.2, 3.8), 2)
        records.append({
            "sku_id":          f"SKU-{sku_id}",
            "product_name":    product,
            "category":        category,
            "cost_price_gbp":  cost,
            "rrp_gbp":         rrp,
            "margin_pct":      round((rrp - cost) / rrp * 100, 1),
            "supplier":        np.random.choice(["Supplier_A","Supplier_B","Supplier_C","Supplier_D"]),
            "lead_time_days":  int(np.random.choice([7, 14, 21, 28, 42])),
            "launch_year":     int(np.random.choice([2008, 2009, 2010])),
            "is_active":       True,
        })
        sku_id += 1

df_erp = spark.createDataFrame(pd.DataFrame(records))
df_erp = df_erp.withColumn("ingested_at", F.current_timestamp())

(df_erp.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.silver_erp_product_master"))

print(f"✅ silver_erp_product_master: {df_erp.count()} SKUs across 8 categories")
display(df_erp)

✅ silver_erp_product_master: 40 SKUs across 8 categories


sku_id,product_name,category,cost_price_gbp,rrp_gbp,margin_pct,supplier,lead_time_days,launch_year,is_active,ingested_at
SKU-1000,Bottle Set,Feeding,19.04,70.85,73.1,Supplier_C,42,2008,true,2026-03-18T10:34:17.983Z
SKU-1001,Steriliser,Feeding,9.97,24.42,59.2,Supplier_C,21,2008,true,2026-03-18T10:34:17.983Z
SKU-1002,Weaning Spoons,Feeding,28.45,94.82,70.0,Supplier_B,42,2009,true,2026-03-18T10:34:17.983Z
SKU-1003,Sippy Cup,Feeding,33.46,123.86,73.0,Supplier_B,28,2008,true,2026-03-18T10:34:17.983Z
SKU-1004,Breast Pump,Feeding,29.13,92.59,68.5,Supplier_A,28,2008,true,2026-03-18T10:34:17.983Z
SKU-1005,Pushchair,Travel,15.59,49.56,68.5,Supplier_B,28,2010,true,2026-03-18T10:34:17.983Z
SKU-1006,Car Seat,Travel,22.43,77.52,71.1,Supplier_C,28,2010,true,2026-03-18T10:34:17.983Z
SKU-1007,Baby Carrier,Travel,44.3,130.54,66.1,Supplier_A,21,2010,true,2026-03-18T10:34:17.983Z
SKU-1008,Travel Cot,Travel,10.58,24.38,56.6,Supplier_D,7,2009,true,2026-03-18T10:34:17.983Z
SKU-1009,Buggy Board,Travel,37.05,99.57,62.8,Supplier_A,14,2010,true,2026-03-18T10:34:17.983Z


## Cell 12 — Generate Synthetic Retailer Funding Table

In [0]:
retailer_rows = [
    ("RET-001", "Boots",         "Retail", 4.5, 2.5, 0.35, 15000, 30),
    ("RET-002", "John Lewis",    "Retail", 3.0, 1.5, 0.25, 10000, 45),
    ("RET-003", "Asda",          "Retail", 5.5, 2.0, 0.50, 20000, 30),
    ("RET-004", "Mothercare",    "Retail", 3.5, 1.0, 0.30,  8000, 60),
    ("RET-005", "Smyths Toys",   "Retail", 4.0, 1.5, 0.40, 12000, 30),
    ("RET-006", "DTC Ecommerce", "DTC",    0.0, 0.0, 0.00,     0,  0),
    ("RET-007", "Amazon UK",     "DTC",    8.0, 0.0, 0.00,     0, 14),
    ("RET-008", "Very.co.uk",    "Retail", 5.0, 2.0, 0.45,  9000, 30),
]

ret_schema = StructType([
    StructField("retailer_id",              StringType(),  False),
    StructField("retailer_name",            StringType(),  False),
    StructField("channel",                  StringType(),  False),
    StructField("rebate_pct",               DoubleType(),  False),
    StructField("settlement_disc_pct",      DoubleType(),  False),
    StructField("scan_allowance_per_unit",  DoubleType(),  False),
    StructField("fixed_annual_funding_gbp", IntegerType(), False),
    StructField("payment_terms_days",       IntegerType(), False),
])

df_funding = spark.createDataFrame(retailer_rows, ret_schema)
df_funding = df_funding.withColumn("ingested_at", F.current_timestamp())

(df_funding.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.silver_retailer_funding"))

print("✅ silver_retailer_funding written")
display(df_funding)

✅ silver_retailer_funding written


retailer_id,retailer_name,channel,rebate_pct,settlement_disc_pct,scan_allowance_per_unit,fixed_annual_funding_gbp,payment_terms_days,ingested_at
RET-001,Boots,Retail,4.5,2.5,0.35,15000,30,2026-03-18T10:34:28.970Z
RET-002,John Lewis,Retail,3.0,1.5,0.25,10000,45,2026-03-18T10:34:28.970Z
RET-003,Asda,Retail,5.5,2.0,0.5,20000,30,2026-03-18T10:34:28.970Z
RET-004,Mothercare,Retail,3.5,1.0,0.3,8000,60,2026-03-18T10:34:28.970Z
RET-005,Smyths Toys,Retail,4.0,1.5,0.4,12000,30,2026-03-18T10:34:28.970Z
RET-006,DTC Ecommerce,DTC,0.0,0.0,0.0,0,0,2026-03-18T10:34:28.970Z
RET-007,Amazon UK,DTC,8.0,0.0,0.0,0,14,2026-03-18T10:34:28.970Z
RET-008,Very.co.uk,Retail,5.0,2.0,0.45,9000,30,2026-03-18T10:34:28.970Z


## Cell 13 — Generate Synthetic Promotional Calendar

In [0]:
promo_rows = [
    ("PRO-001","RET-001","Boots Baby Event",       "2010-01-15","2010-01-28",20.0,"Scan",    3000),
    ("PRO-002","RET-003","Asda Baby Week",         "2010-02-01","2010-02-07",25.0,"Scan",    5000),
    ("PRO-003","RET-002","JL Spring Baby Fair",    "2010-03-01","2010-03-14",15.0,"Feature", 2500),
    ("PRO-004","RET-001","Boots Easter Promo",     "2010-04-01","2010-04-10",20.0,"Scan",    2800),
    ("PRO-005","RET-005","Smyths Summer Sale",     "2010-06-01","2010-06-30",30.0,"Markdown",4000),
    ("PRO-006","RET-003","Asda Back to School",    "2010-08-15","2010-08-31",20.0,"Scan",    4500),
    ("PRO-007","RET-001","Boots Baby Event Q3",    "2010-09-01","2010-09-14",20.0,"Scan",    3200),
    ("PRO-008","RET-002","JL Christmas Nursery",   "2010-11-01","2010-11-30",15.0,"Feature", 3500),
    ("PRO-009","RET-003","Asda Black Friday Baby", "2010-11-26","2010-11-27",35.0,"Markdown",6000),
    ("PRO-010","RET-001","Boots Xmas Event",       "2010-12-01","2010-12-24",20.0,"Scan",    4000),
    ("PRO-011","RET-006","DTC New Year Sale",      "2011-01-01","2011-01-07",25.0,"Markdown",1000),
    ("PRO-012","RET-007","Amazon Lightning Deal",  "2011-02-14","2011-02-14",30.0,"Markdown", 500),
]

promo_schema = StructType([
    StructField("promo_id",        StringType(),  False),
    StructField("retailer_id",     StringType(),  False),
    StructField("promo_name",      StringType(),  False),
    StructField("start_date",      StringType(),  False),
    StructField("end_date",        StringType(),  False),
    StructField("disc_pct",        DoubleType(),  False),
    StructField("promo_type",      StringType(),  False),
    StructField("trade_spend_gbp", IntegerType(), False),
])

df_promos = spark.createDataFrame(promo_rows, promo_schema)
df_promos = (df_promos
    .withColumn("start_date",  F.to_date("start_date"))
    .withColumn("end_date",    F.to_date("end_date"))
    .withColumn("ingested_at", F.current_timestamp()))

(df_promos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.silver_promo_calendar"))

print("✅ silver_promo_calendar written")
display(df_promos)

✅ silver_promo_calendar written


promo_id,retailer_id,promo_name,start_date,end_date,disc_pct,promo_type,trade_spend_gbp,ingested_at
PRO-001,RET-001,Boots Baby Event,2010-01-15,2010-01-28,20.0,Scan,3000,2026-03-18T10:34:35.503Z
PRO-002,RET-003,Asda Baby Week,2010-02-01,2010-02-07,25.0,Scan,5000,2026-03-18T10:34:35.503Z
PRO-003,RET-002,JL Spring Baby Fair,2010-03-01,2010-03-14,15.0,Feature,2500,2026-03-18T10:34:35.503Z
PRO-004,RET-001,Boots Easter Promo,2010-04-01,2010-04-10,20.0,Scan,2800,2026-03-18T10:34:35.503Z
PRO-005,RET-005,Smyths Summer Sale,2010-06-01,2010-06-30,30.0,Markdown,4000,2026-03-18T10:34:35.503Z
PRO-006,RET-003,Asda Back to School,2010-08-15,2010-08-31,20.0,Scan,4500,2026-03-18T10:34:35.503Z
PRO-007,RET-001,Boots Baby Event Q3,2010-09-01,2010-09-14,20.0,Scan,3200,2026-03-18T10:34:35.503Z
PRO-008,RET-002,JL Christmas Nursery,2010-11-01,2010-11-30,15.0,Feature,3500,2026-03-18T10:34:35.503Z
PRO-009,RET-003,Asda Black Friday Baby,2010-11-26,2010-11-27,35.0,Markdown,6000,2026-03-18T10:34:35.503Z
PRO-010,RET-001,Boots Xmas Event,2010-12-01,2010-12-24,20.0,Scan,4000,2026-03-18T10:34:35.503Z


## Cell 14 — UK Region Lookup Table

In [0]:
uk_region_rows = [
    ("London","London"),("Manchester","North West"),("Birmingham","West Midlands"),
    ("Leeds","Yorkshire"),("Glasgow","Scotland"),("Liverpool","North West"),
    ("Bristol","South West"),("Sheffield","Yorkshire"),("Edinburgh","Scotland"),
    ("Cardiff","Wales"),("Leicester","East Midlands"),("Nottingham","East Midlands"),
    ("Newcastle","North East"),("Southampton","South East"),("Brighton","South East"),
    ("Oxford","South East"),("Cambridge","East of England"),("Norwich","East of England"),
    ("Exeter","South West"),("Belfast","Northern Ireland"),("Aberdeen","Scotland"),
    ("Coventry","West Midlands"),("Reading","South East"),("Milton Keynes","South East"),
    ("Stoke","West Midlands"),("Derby","East Midlands"),("Plymouth","South West"),
    ("Wolverhampton","West Midlands"),("Swansea","Wales"),("York","Yorkshire"),
    ("Other","Other UK"),
]

region_schema = StructType([
    StructField("city",      StringType(), False),
    StructField("uk_region", StringType(), False),
])

df_regions = spark.createDataFrame(uk_region_rows, region_schema)
df_regions = df_regions.withColumn("ingested_at", F.current_timestamp())

(df_regions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.silver_uk_region_lookup"))

print("✅ silver_uk_region_lookup written")
display(df_regions)

✅ silver_uk_region_lookup written


city,uk_region,ingested_at
London,London,2026-03-18T10:34:44.772Z
Manchester,North West,2026-03-18T10:34:44.772Z
Birmingham,West Midlands,2026-03-18T10:34:44.772Z
Leeds,Yorkshire,2026-03-18T10:34:44.772Z
Glasgow,Scotland,2026-03-18T10:34:44.772Z
Liverpool,North West,2026-03-18T10:34:44.772Z
Bristol,South West,2026-03-18T10:34:44.772Z
Sheffield,Yorkshire,2026-03-18T10:34:44.772Z
Edinburgh,Scotland,2026-03-18T10:34:44.772Z
Cardiff,Wales,2026-03-18T10:34:44.772Z


## Cell 15 — Quality Check: All Tables

In [0]:
all_tables = [
    "bronze_pos_raw",
    "bronze_trends_baby_timeseries",
    "bronze_trends_baby_regions",
    "bronze_trends_nursery_timeseries",
    "bronze_trends_nursery_regions",
    "bronze_ons_retail_index",
    "bronze_ons_retail_timeseries",
    "silver_erp_product_master",
    "silver_retailer_funding",
    "silver_promo_calendar",
    "silver_uk_region_lookup",
]

print("=" * 60)
print(f"  {'TABLE':<44} {'ROWS':>10}")
print("=" * 60)

all_ok = True
for t in all_tables:
    try:
        n = spark.sql(f"SELECT COUNT(*) AS n FROM {DATABASE}.{t}").collect()[0]["n"]
        print(f"  ✅  {t:<44} {n:>10,}")
    except Exception as e:
        all_ok = False
        print(f"  ❌  {t:<44}  ERROR: {e}")

print("=" * 60)
if all_ok:
    print("  🎉 MODULE 7 COMPLETE — All 11 tables loaded successfully!")
    print("  👉 Next step: Module 1 — Silver Cleaning & Commercial P&L")
else:
    print("  ⚠️  Some tables failed — review errors above")

  TABLE                                              ROWS
  ✅  bronze_pos_raw                                1,067,371
  ✅  bronze_trends_baby_timeseries                       262
  ✅  bronze_trends_baby_regions                            4
  ✅  bronze_trends_nursery_timeseries                    262
  ✅  bronze_trends_nursery_regions                         4
  ✅  bronze_ons_retail_index                             916
  ✅  bronze_ons_retail_timeseries                        457
  ✅  silver_erp_product_master                            40
  ✅  silver_retailer_funding                               8
  ✅  silver_promo_calendar                                12
  ✅  silver_uk_region_lookup                              31
  🎉 MODULE 7 COMPLETE — All 11 tables loaded successfully!
  👉 Next step: Module 1 — Silver Cleaning & Commercial P&L
